# FLARE — Early Detection of Low-Rate Flow-Table Overflow (LOFT) Attacks in SDN

**One-click reproducible notebook.** Runs the full FLARE pipeline: a realistic
SDN switch + traffic simulator, the detector stack (FloRa-style baseline vs
FLARE), early-warning probing detection, time-to-overflow forecasting,
explainability, and mitigation — rendering every figure inline.

> Runtime → *Run all*. No GPU or special hardware required.

## 1. Set up — clone the repo and install dependencies

In [ ]:
import os, sys

REPO = 'https://github.com/prakhar443/project_FLARE.git'
BRANCH = 'claude/affectionate-archimedes-rjk02'
if not os.path.isdir('project_flare'):
    !git clone --branch $BRANCH --depth 1 $REPO project_flare
%cd project_flare
!pip -q install -r requirements.txt
sys.path.insert(0, os.getcwd())

## 2. Configure the scenario

`SimConfig` holds the switch (capacity, idle timeout), benign-traffic, and LOFT
attack parameters. Everything is seeded for exact reproducibility.

In [ ]:
from flare.config import SimConfig
from flare.simulator import SDNSimulator, feature_columns

cfg = SimConfig(seed=7)
print('capacity      :', cfg.capacity, 'entries')
print('idle_timeout  :', cfg.idle_timeout, 's')
print('probe phase   :', cfg.probe_start, '->', cfg.probe_start + cfg.probe_duration, 's')
print('attack starts :', cfg.attack_start, 's')

## 3. Simulate the attack on an *undefended* switch

The low-rate attack slowly fills the flow table until legitimate flows can no
longer be installed (overflow).

In [ ]:
import matplotlib.pyplot as plt

df = SDNSimulator(cfg).run()
overflow_t = df.attrs['overflow_t']
print('overflow time (table effectively full):', overflow_t, 's')

fig, ax = plt.subplots(figsize=(10,4.5))
ax.plot(df['t'], df['occupancy'], color='#c0392b', label='flow-table occupancy')
ax.axhline(cfg.capacity, ls='--', color='black', label='capacity')
ax.axvspan(cfg.probe_start, cfg.probe_start+cfg.probe_duration, color='#f1c40f', alpha=.25, label='probing phase')
ax.axvspan(cfg.attack_start, cfg.duration, color='#e74c3c', alpha=.12, label='attack phase')
if overflow_t: ax.axvline(overflow_t, color='purple', ls=':', label=f'overflow @ {overflow_t:.0f}s')
ax.set_xlabel('time (s)'); ax.set_ylabel('entries'); ax.legend(loc='upper left', fontsize=8)
ax.set_title('LOFT attack on an undefended SDN switch'); plt.show()

## 4. Inspect the per-window features

FLARE consumes these windowed features. Note how flow-table *growth* and
*churn* shift during the probing and attack phases while overall traffic volume
stays low — the essence of a stealthy attack.

In [ ]:
df[['t','occupancy','occupancy_ratio','occ_growth_ewma','miss_rate','short_lived_rate','phase']].iloc[::25]

## 5. Run every detector and compare

`evaluate` streams the scenario one window at a time through the FloRa-style
baseline and the full FLARE stack — exactly as it would run live on a switch.

In [ ]:
from flare.evaluation import evaluate, compare_methods, false_positive_rate
from flare.dataset import generate_benign_only

res = evaluate(df, cfg)
for k, v in res.summary().items():
    print(f'{k:32s}: {v}')

In [ ]:
benign = generate_benign_only(cfg, n_runs=3)
table = compare_methods(df, cfg, benign_df=benign)
print('overflow_t       :', table.attrs['overflow_t'], 's')
print('forecast_error_s :', round(table.attrs['forecast_error_s'], 1), 's')
print('false_pos_rate   :', table.attrs['false_positive_rate'])
table

## 6. Detection timeline — FLARE warns earlier than the baseline

FLARE's early-warning and core detectors fire during the probing phase, far
ahead of both the table overflow and the reactive FloRa-style baseline.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10,7), sharex=True, gridspec_kw={'height_ratios':[2,1]})
ax1.plot(df['t'], df['occupancy'], color='#2c3e50', label='occupancy')
ax1.axhline(cfg.capacity, ls='--', color='black', label='capacity')
for t,c,l in [(res.probe_t,'#f39c12','FLARE early-warning'),(res.flare_t,'#27ae60','FLARE core'),
              (res.baseline_t,'#c0392b','baseline'),(res.overflow_t,'purple','overflow')]:
    if t: ax1.axvline(t, color=c, lw=2, ls=(':' if l=='overflow' else '-'), label=f'{l} @ {t:.0f}s')
ax1.set_ylabel('entries'); ax1.legend(loc='upper left', fontsize=8)
ax1.set_title('Detection timeline: FLARE warns earlier than the baseline')
score_t = df.loc[df['t']>=cfg.transient_skip,'t'].values[:len(res.scores)]
ax2.plot(score_t, res.scores, color='#8e44ad', label='FLARE anomaly score')
ax2.axhline(3.0, ls='--', color='gray', label='threshold')
ax2.set_xlabel('time (s)'); ax2.set_ylabel('score'); ax2.legend(loc='upper left', fontsize=8); plt.show()

## 7. Time-to-overflow forecast

FLARE projects the occupancy trend to an actionable ETA that converges on the
true overflow time.

In [ ]:
fc = [(t, t+eta) for (t, eta, _s) in res.forecast_curve if eta is not None]
if fc:
    ts, pred = zip(*fc)
    fig, ax = plt.subplots(figsize=(10,4.5))
    ax.plot(ts, pred, color='#2980b9', label='predicted overflow time')
    ax.plot(ts, ts, ls=':', color='gray', label='present (y=x)')
    if res.overflow_t: ax.axhline(res.overflow_t, ls='--', color='purple', label=f'actual overflow @ {res.overflow_t:.0f}s')
    ax.set_xlabel('time now (s)'); ax.set_ylabel('forecast overflow time (s)')
    ax.set_title('FLARE time-to-overflow forecast'); ax.legend(loc='upper right', fontsize=8); plt.show()
    print('forecast error at 60%% occupancy:', round(res.forecast_error or float('nan'), 1), 's')

## 8. Explainability — every alert comes with a reason

FLARE names the dominant anomalous features behind each alert so operators can
trust and act on it.

In [ ]:
for a in res.flare_alerts[:5]:
    print(f't={a.t:6.0f}s  score={a.score:5.2f}  ->  {a.reason}')
print('--- early-warning (probing) alerts ---')
for a in res.probe_alerts[:5]:
    print(f't={a.t:6.0f}s  score={a.score:5.2f}  ->  {a.reason}')

## 9. Mitigation — FLARE keeps the table from overflowing

On confirmation FLARE selectively evicts low-activity flows and hardens the idle
timeout. Re-simulating with mitigation enabled at FLARE's detection time keeps
occupancy bounded instead of overflowing.

In [ ]:
from flare.mitigation import MitigationEngine

activate_t = res.flare_t or res.probe_t or cfg.attack_start
mit = MitigationEngine(cfg).simulate_with_mitigation(activate_t)
fig, ax = plt.subplots(figsize=(10,4.5))
ax.plot(df['t'], df['occupancy'], color='#c0392b', label='no defense')
ax.plot(mit['t'], mit['occupancy'], color='#27ae60', label='FLARE mitigation')
ax.axhline(cfg.capacity, ls='--', color='black', label='capacity')
ax.axvline(activate_t, color='#27ae60', ls=':', label=f'mitigation @ {activate_t:.0f}s')
ax.set_xlabel('time (s)'); ax.set_ylabel('entries')
ax.set_title('FLARE mitigation keeps the flow table from overflowing'); ax.legend(loc='upper left', fontsize=8); plt.show()

## 10. Labeled dataset & false-positive analysis

Generate the multi-run labeled dataset and confirm FLARE raises no false alarms
on attack-free traffic.

In [ ]:
from flare.dataset import generate_dataset
ds = generate_dataset(cfg, n_runs=5)
print('dataset shape:', ds.shape, '| phases:', dict(ds['phase'].value_counts()))
print('false-positive rate (attack-free):', false_positive_rate(benign, cfg))
ds.head()

## 11. (Optional) Run the full experiment script & test suite

In [ ]:
!python scripts/run_experiments.py --outdir figures
!python -m pytest -q

---
## Summary

On the default scenario FLARE:
- detects the attack **~125 s earlier** than the FloRa-style baseline,
- warns **~150 s before** the flow table overflows (vs ~25 s for the baseline),
- forecasts the overflow time to within **~14 s**,
- raises **0 false positives** on attack-free traffic, and
- explains every alert and bounds occupancy via mitigation.

See `docs/paper_outline.md` for the IEEE paper structure.